In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import log_loss, make_scorer


traindata = pd.read_csv('/kaggle/input/project-1-me-4127-e/train_data.csv')
traindata.columns = traindata.columns.astype(str)
print("Train data loaded")

traindata = traindata.drop(columns=['id'])

features = traindata.drop(columns=['target'])
target = traindata['target']

catcolumns = features.select_dtypes(include=['object']).columns.tolist()
numcolumns = features.select_dtypes(include=['float64', 'int64']).columns.tolist()

print(f"cat features= {catcolumns}")
print(f"numerical= {numcolumns}")

encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
encodedcat = encoder.fit_transform(features[catcolumns])
print("cat features encoded.")

encodeddata = pd.DataFrame(encodedcat, columns=encoder.get_feature_names_out(catcolumns))

scaler = StandardScaler()
scalednum = scaler.fit_transform(features[numcolumns])

trainfeatures = pd.concat([encodeddata.reset_index(drop=True), pd.DataFrame(scalednum, columns=numcolumns).reset_index(drop=True)], axis=1)

model = LogisticRegression(max_iter=1000)

loglossscorer = make_scorer(log_loss, needs_proba=True, greater_is_better=False)

print("CV..")
cvscores = cross_val_score(model, trainfeatures, target, cv=5, scoring=loglossscorer)

print(f"CV Log Loss Score: {-cvscores}")
print(f"Mean Log Loss: {-cvscores.mean()}")

model.fit(trainfeatures, target)


testdata = pd.read_csv('/kaggle/input/project-1-me-4127-e/test_data_file.csv')
print("test data loaded")

testdata = testdata.drop(columns=['id'])

testfeatures = testdata.drop(columns=['target'], errors='ignore')

testencodedcat = encoder.transform(testfeatures[catcolumns])
testencodeddata = pd.DataFrame(testencodedcat, columns=encoder.get_feature_names_out(catcolumns))

testscalednum = scaler.transform(testfeatures[numcolumns])

testfinal = pd.concat([testencodeddata.reset_index(drop=True), pd.DataFrame(testscalednum, columns=numcolumns).reset_index(drop=True)], axis=1)

predictions = model.predict_proba(testfinal)[:, 1]

submission = pd.DataFrame({
    'id': pd.read_csv('/kaggle/input/project-1-me-4127-e/test_data_file.csv')['id'],
    'target': predictions
})

submission.to_csv('submission.csv', index=False)
print("csv file created")